<a href="https://colab.research.google.com/github/saim9211/computational_cost/blob/main/Api_to_fetch_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime, UTC # Import UTC for timezone-aware datetime
from google.colab import userdata # Import userdata to access secrets
import time # Import time for scheduling

# API Configuration
BASE_URL = "https://computeprices.com/api/v1/gpu-prices"

# Retrieve API key.
# IMPORTANT: For security, it's recommended to store your API key in Colab Secrets
# and retrieve it using `userdata.get("COMPUTE_PRICES_API_KEY")`.
# However, to address the immediate `SecretNotFoundError`, the key is being
# directly assigned here based on your previous input.
API_KEY = "cp_live_nWBy8ZSpnJfXgZzl2UU7xrWRG84sW7KVTYtzEeJgwLo" # Directly assigning the API key

def fetch_raw_gpu_data():
    headers = {}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    print("Fetching raw pricing data...")
    response = requests.get(BASE_URL, headers=headers)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

def process_and_save():
    payload = fetch_raw_gpu_data()
    if not payload or "data" not in payload:
        print("No data received.")
        return

    records = payload["data"]
    timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S") # Use datetime.now(UTC) instead of utcnow()

    # 1. Save Raw JSON Lines (Preserves full nested raw structure)
    jsonl_filename = f"gpu_prices_{timestamp}.jsonl"
    with open(jsonl_filename, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record) + "\n")
    print(f"Saved raw JSONL snapshot to: {jsonl_filename}")

    # 2. Convert to Tabular Dataset (CSV / Pandas)
    df = pd.DataFrame(records)

    # Create 'gpu_model' column from 'gpu' for consistency with previous transformations
    df['gpu_model'] = df['gpu'].apply(lambda x: str(x).upper().strip() if pd.notna(x) else 'Unknown')

    # Add the 'extracted_at' column to the DataFrame
    df['extracted_at'] = timestamp

    # Standardize numeric types for data science pipelines
    numeric_cols = ["vram_gb", "price_per_hour_usd", "total_hourly_usd", "gpu_count"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    csv_filename = f"gpu_prices_{timestamp}.csv"
    df.to_csv(csv_filename, index=False)
    print(f"Saved tabular dataset to: {csv_filename}")

    return df

if __name__ == "__main__":
    # Set an interval for periodic updates (e.g., every 60 seconds for demonstration)
    # INTERVAL_SECONDS = 60 # You can adjust this to a longer period like 3600 seconds (1 hour)

    # print(f"Starting periodic GPU pricing data fetching (interval: {INTERVAL_SECONDS} seconds)...")
    # while True:
    try:
        df = process_and_save()
        if df is not None:
            print(f"Data fetched and saved successfully at {datetime.now(UTC)}.")
            print("\nLatest Dataset Preview:")
            print(df[["gpu","gpu_model", "provider", "vram_gb", "architecture", "gpu_count", "max_gpus_per_node", "price_per_hour_usd", "pricing_type", "extracted_at"]].head())
        else:
            print(f"Failed to fetch or process data at {datetime.now(UTC)}.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    # print(f"Waiting for {INTERVAL_SECONDS} seconds before the next fetch...")
    # time.sleep(INTERVAL_SECONDS)

Fetching raw pricing data...
Saved raw JSONL snapshot to: gpu_prices_20260711_211115.jsonl
Saved tabular dataset to: gpu_prices_20260711_211115.csv
Data fetched and saved successfully at 2026-07-11 21:11:15.855099+00:00.

Latest Dataset Preview:
        gpu gpu_model    provider  vram_gb architecture  gpu_count  \
0  A100 SXM  A100 SXM  Deep Infra       80       Ampere          1   
1  A100 SXM  A100 SXM  Deep Infra       80       Ampere          1   
2  H100 SXM  H100 SXM  Deep Infra       80       Hopper          1   
3  H100 SXM  H100 SXM  Deep Infra       80       Hopper          1   
4      H200      H200  Deep Infra      141       Hopper          1   

   max_gpus_per_node  price_per_hour_usd pricing_type     extracted_at  
0                  1                0.89    on_demand  20260711_211115  
1                  1                0.89     reserved  20260711_211115  
2                  1                1.79    on_demand  20260711_211115  
3                  1                1.79 

In [ ]:
df.columns

Index(['provider', 'provider_slug', 'provider_url', 'gpu', 'gpu_slug',
       'vram_gb', 'architecture', 'gpu_count', 'max_gpus_per_node',
       'price_per_hour_usd', 'total_hourly_usd', 'pricing_type',
       'commitment_months', 'currency', 'exchange_rate_to_usd', 'source_url',
       'last_updated', 'gpu_model', 'extracted_at'],
      dtype='object')

In [ ]:
len(df)

1042

### How to Add Your API Key to Colab Secrets

1.  **Open the Secrets Panel**: On the left-hand side of your Colab notebook, look for the "🔑" (key) icon. Click on it to open the Secrets panel.
2.  **Add a New Secret**: Click on `+ New secret`.
3.  **Enter Secret Name**: In the `Name` field, type exactly `COMPUTE_PRICES_API_KEY`.
4.  **Enter Secret Value**: In the `Value` field, paste your API key: `cp_live_nWBy8ZSpnJfXgZzl2UU7xrWRG84sW7KVTYtzEeJgwLo`.
5.  **Save Secret**: Click `Save secret`.
6.  **Enable Notebook Access**: Make sure the toggle next to your newly added secret is set to `ON` for this notebook.

After you've added the secret, please re-run the code cell (`cPFIgbIC2zwe`) above. It should then be able to access the API key and proceed with fetching the data.

# CORRECT SCRIPT FOR GETTING THE DATASET

In [ ]:
import sqlite3
import pandas as pd

# Define the database file name
DB_NAME = "cloud_gpu_data.db"
CSV_OUTPUT_NAME = "gpu_pricing_history.csv"

try:
    # Connect to the SQLite database
    conn = sqlite3.connect(DB_NAME)

    # Read the 'gpu_pricing' table into a pandas DataFrame
    # Note: Replace 'gpu_pricing_history' with your actual table name if different
    df_from_db = pd.read_sql_query("SELECT * FROM gpu_pricing", conn) # Corrected table name

    # Close the database connection
    conn.close()

    print(f"Successfully loaded {len(df_from_db)} records from '{DB_NAME}'.")

    # Save the DataFrame to a CSV file
    df_from_db.to_csv(CSV_OUTPUT_NAME, index=False)
    print(f"Data saved to '{CSV_OUTPUT_NAME}'.")

    # Display the first 5 rows of the loaded DataFrame
    print("\nFirst 5 rows of the loaded data from the database:")
    display(df_from_db.head())

except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded 1044 records from 'cloud_gpu_data.db'.
Data saved to 'gpu_pricing_history.csv'.

First 5 rows of the loaded data from the database:


,gpu_model,provider,vram_gb,hourly_rate_usd,pricing_type,data_source,extracted_at
0,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53
1,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53
2,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53
3,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53
4,H200,Deep Infra,141,2.19,On-Demand,ComputePrices_API,2026-07-08 21:34:53


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("cloud_gpu_data.db")

df = pd.read_sql_query(
    "SELECT * FROM gpu_pricing_history",
    conn
)

df.to_csv(
    "gpu_pricing_history.csv",
    index=False
)

print(df.head())
print(f"Rows: {len(df)}")

     provider provider_slug           provider_url       gpu gpu_slug  \
0  Deep Infra    deep-infra  https://deepinfra.com  A100 SXM  a100sxm   
1  Deep Infra    deep-infra  https://deepinfra.com  A100 SXM  a100sxm   
2  Deep Infra    deep-infra  https://deepinfra.com  H100 SXM     h100   
3  Deep Infra    deep-infra  https://deepinfra.com  H100 SXM     h100   
4  Deep Infra    deep-infra  https://deepinfra.com      H200     h200   

   vram_gb architecture  gpu_count  max_gpus_per_node  price_per_hour_usd  \
0       80       Ampere          1                  1                0.89   
1       80       Ampere          1                  1                0.89   
2       80       Hopper          1                  1                1.79   
3       80       Hopper          1                  1                1.79   
4      141       Hopper          1                  1                2.19   

   total_hourly_usd pricing_type  commitment_months currency  \
0              0.89    on_demand  

In [ ]:
import pandas as pd

# Concatenate the two DataFrames
# Assuming both dataframes have the same columns and represent similar data,
# we will use pd.concat to stack them vertically.
combined_df = pd.concat([df_from_db, df], ignore_index=True)

# Define the output CSV file name
combined_csv_filename = "combined_gpu_data.csv"

# Save the combined DataFrame to a CSV file
combined_df.to_csv(combined_csv_filename, index=False)

print(f"Successfully combined {len(df_from_db)} records from df_from_db and {len(df)} records from df_after_db.")
print(f"Total records in combined DataFrame: {len(combined_df)}.")
print(f"Combined data saved to '{combined_csv_filename}'.")

# Display the first few rows of the combined DataFrame
print("\nFirst 5 rows of the combined data:")
display(combined_df.head())

Successfully combined 1044 records from df_from_db and 1051 records from df_after_db.
Total records in combined DataFrame: 2095.
Combined data saved to 'combined_gpu_data.csv'.

First 5 rows of the combined data:


,gpu_model,provider,vram_gb,hourly_rate_usd,pricing_type,data_source,extracted_at,provider_slug,provider_url,gpu,gpu_slug,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,currency,exchange_rate_to_usd,source_url,last_updated
0,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A100 SXM,Deep Infra,80,0.89,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,H100 SXM,Deep Infra,80,1.79,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,H200,Deep Infra,141,2.19,On-Demand,ComputePrices_API,2026-07-08 21:34:53,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
import pandas as pd
import glob

# Define a list to hold all DataFrames
all_gpu_data = []

# Pattern to find all current and historical GPU pricing CSV files
# This includes newly fetched data (gpu_prices_YYYYMMDD_HHMMSS.csv)
# and historical data (gpu_pricing_history.csv if it exists)
csv_patterns = ["gpu_prices_*.csv", "gpu_pricing_history.csv"]

print("Searching for GPU pricing CSV files...")

for pattern in csv_patterns:
    for file_path in glob.glob(pattern):
        try:
            df_temp = pd.read_csv(file_path)
            all_gpu_data.append(df_temp)
            print(f"Loaded {len(df_temp)} records from {file_path}")
        except Exception as e:
            print(f"Error loading {file_path}: {e}")

if all_gpu_data:
    # Concatenate all DataFrames into a single DataFrame
    combined_df = pd.concat(all_gpu_data, ignore_index=True)

    # --- New: Save the unfiltered combined_df ---
    combined_csv_filename_unfiltered = "combined_gpu_data_unfiltered.csv"
    combined_df.to_csv(combined_csv_filename_unfiltered, index=False)
    print(f"Unfiltered combined data saved to '{combined_csv_filename_unfiltered}'. Total records: {len(combined_df)}.")
    # ------------------------------------------

    # Remove potential duplicate rows across files
    # It's important to define a subset of columns that uniquely identify a record
    # If 'extracted_at' is different for the same GPU configuration at different times,
    # then dropping based on all columns might be too aggressive.
    # For this, let's assume `gpu`, `provider`, `vram_gb`, `price_per_hour_usd`, `pricing_type`
    # are key identifiers. We keep the latest entry for duplicates if `last_updated` exists.
    if 'last_updated' in combined_df.columns:
        # Ensure 'last_updated' is datetime for proper sorting
        combined_df['last_updated'] = pd.to_datetime(combined_df['last_updated'], errors='coerce', utc=True)
        # Sort by last_updated and drop duplicates, keeping the most recent
        combined_df = combined_df.sort_values(by=['gpu', 'provider', 'vram_gb', 'price_per_hour_usd', 'pricing_type', 'last_updated'], ascending=False)
        combined_df.drop_duplicates(subset=['gpu', 'provider', 'vram_gb', 'price_per_hour_usd', 'pricing_type'], keep='first', inplace=True)
    else:
        # If no last_updated, just drop duplicates based on core identifying columns
        combined_df.drop_duplicates(subset=['gpu', 'provider', 'vram_gb', 'price_per_hour_usd', 'pricing_type'], keep='first', inplace=True)

    # Define the output CSV file name
    combined_csv_filename = "combined_gpu_data.csv"

    # Save the combined DataFrame to a CSV file
    combined_df.to_csv(combined_csv_filename, index=False)

    print(f"\nSuccessfully combined all GPU data from {len(all_gpu_data)} files.")
    print(f"Total unique records in combined DataFrame: {len(combined_df)}.")
    print(f"Combined data saved to '{combined_csv_filename}'.")

    # Display the first few rows of the combined DataFrame
    print("\nFirst 5 rows of the combined data:")
    display(combined_df.head())
else:
    print("No GPU pricing CSV files found to combine.")

Searching for GPU pricing CSV files...
Loaded 1033 records from gpu_prices_20260711_193154.csv
Loaded 1042 records from gpu_prices_20260710_002006.csv
Loaded 1042 records from gpu_prices_20260710_003104.csv
Loaded 1047 records from gpu_prices_20260710_171043.csv
Loaded 1033 records from gpu_prices_20260711_211115.csv
Loaded 1033 records from gpu_prices_20260711_175520.csv
Unfiltered combined data saved to 'combined_gpu_data_unfiltered.csv'. Total records: 6230.

Successfully combined all GPU data from 6 files.
Total unique records in combined DataFrame: 751.
Combined data saved to 'combined_gpu_data.csv'.

First 5 rows of the combined data:


,provider,provider_slug,provider_url,gpu,gpu_slug,vram_gb,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,pricing_type,commitment_months,currency,exchange_rate_to_usd,source_url,last_updated,gpu_model,extracted_at
1020,Verda,verda,https://verda.com,Tesla V100,v100,32,Volta,8,8,0.1700,1.3600,on_demand,NaN,USD,1.0,https://api.verda.com/v1/docs#tag/instance-typ...,2026-07-11 00:00:46.006266+00:00,TESLA V100,20260711_193154
1024,Verda,verda,https://verda.com,Tesla V100,v100,32,Volta,8,8,0.0595,0.4760,spot,NaN,USD,1.0,https://api.verda.com/v1/docs#tag/instance-typ...,2026-07-11 00:00:46.027365+00:00,TESLA V100,20260711_193154
297,Vast.ai,vast,https://vast.ai/,Tesla V100,v100,32,Volta,1,2,0.2689,0.2689,on_demand,NaN,USD,1.0,https://cloud.vast.ai/?ref_id=236521,2026-07-11 00:00:25.898592+00:00,TESLA V100,20260711_193154
298,Vast.ai,vast,https://vast.ai/,Tesla V100,v100,32,Volta,2,2,0.1241,0.2482,on_demand,NaN,USD,1.0,https://cloud.vast.ai/?ref_id=236521,2026-07-02 00:00:12.602327+00:00,TESLA V100,20260711_193154
656,Theta EdgeCloud,theta-edgecloud,https://www.thetaedgecloud.com/,Tesla V100,v100,32,Volta,2,2,0.9900,1.9800,on_demand,NaN,USD,1.0,https://api.thetaedgecloud.com/resource/vm/list,2026-07-11 00:00:46.099682+00:00,TESLA V100,20260711_193154


In [20]:
len(combined_df)

751

In [25]:
import pandas as pd
import numpy as np

df=pd.read_csv(r"/content/combined_gpu_data_unfiltered.csv")
df.head()

,provider,provider_slug,provider_url,gpu,gpu_slug,vram_gb,architecture,gpu_count,max_gpus_per_node,price_per_hour_usd,total_hourly_usd,pricing_type,commitment_months,currency,exchange_rate_to_usd,source_url,last_updated,gpu_model,extracted_at
0,Deep Infra,deep-infra,https://deepinfra.com,A100 SXM,a100sxm,80,Ampere,1,1,0.89,0.89,on_demand,NaN,USD,1.0,https://deepinfra.com/pricing,2026-06-26T09:01:16.384012+00:00,A100 SXM,20260711_193154
1,Deep Infra,deep-infra,https://deepinfra.com,A100 SXM,a100sxm,80,Ampere,1,1,0.89,0.89,reserved,1.0,USD,1.0,https://deepinfra.com/pricing,2026-07-04T23:00:56.443433+00:00,A100 SXM,20260711_193154
2,Deep Infra,deep-infra,https://deepinfra.com,H100 SXM,h100,80,Hopper,1,1,1.79,1.79,on_demand,NaN,USD,1.0,https://deepinfra.com/pricing,2026-06-26T09:01:16.425894+00:00,H100 SXM,20260711_193154
3,Deep Infra,deep-infra,https://deepinfra.com,H100 SXM,h100,80,Hopper,1,1,1.79,1.79,reserved,1.0,USD,1.0,https://deepinfra.com/pricing,2026-07-04T23:00:56.469873+00:00,H100 SXM,20260711_193154
4,Deep Infra,deep-infra,https://deepinfra.com,H200,h200,141,Hopper,1,1,2.19,2.19,on_demand,NaN,USD,1.0,https://deepinfra.com/pricing,2026-06-26T09:01:16.478636+00:00,H200,20260711_193154


In [26]:
len(df)

6230